# Notebook 03: Paralelismo, Benchmarks y Patrones Avanzados

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/03_paralelismo_benchmarks.ipynb)

**Módulo 16 — Clase 3**

Este notebook acompaña los archivos `04b_asyncio_patrones.md`, `05_paralelismo.md`, `06_librerias_y_decision.md`.

Secciones **[EN CLASE]** se trabajan durante la sesión.  
Secciones **[TAREA]** se completan después.

---

In [ ]:
import asyncio
import time
import threading
import os
import sys
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

print(f'Python {sys.version}')
print(f'CPU cores disponibles: {os.cpu_count()}')

## [EN CLASE] Sección 1: create_task vs gather — trabajo intermedio "gratis"

**Hipótesis:** con `create_task`, podemos ejecutar trabajo independiente mientras una tarea espera.
El trabajo intermedio debe ocurrir "gratis" — durante el wait de la tarea lanzada.

In [ ]:
async def tarea_io(nombre: str, duracion: float) -> str:
    """Simula I/O-bound: wait(τᵢ) ≠ ∅"""
    await asyncio.sleep(duracion)
    return f"{nombre} completada"

def trabajo_cpu_ligero(n: int) -> int:
    """Trabajo CPU-bound ligero: exec(τⱼ) mientras τᵢ espera"""
    return sum(range(n))

# --- Comparación: gather vs create_task ---

# gather: barrera — espera TODAS antes de continuar
print("=== gather: barrera ===")
t0 = time.perf_counter()
r1, r2 = await asyncio.gather(
    tarea_io("A", 1.0),
    tarea_io("B", 1.0),
)
t_gather = time.perf_counter() - t0
print(f"Resultados: {r1}, {r2}")
print(f"Tiempo: {t_gather:.2f}s (esperado ~1s)")
print()

# create_task: lanzar y continuar con trabajo intermedio
print("=== create_task + trabajo intermedio ===")
t0 = time.perf_counter()

# ① Lanzar τᵢ en background
tarea_a = asyncio.create_task(tarea_io("A", 1.0))

# ② Trabajo independiente DURANTE el wait de A
t_trabajo = time.perf_counter()
resultado_cpu = trabajo_cpu_ligero(5_000_000)   # ~200ms de CPU
t_trabajo = time.perf_counter() - t_trabajo
print(f"  Trabajo CPU completado en {t_trabajo:.2f}s (durante wait de A)")

# ③ Punto de dependencia: necesitamos el resultado de A
resultado_a = await tarea_a
t_create_task = time.perf_counter() - t0

print(f"  {resultado_a}")
print(f"Tiempo total: {t_create_task:.2f}s (esperado ~max(1s, tiempo_cpu))")
print()
print(f"Observa: el trabajo CPU ({t_trabajo:.2f}s) fue 'gratis' — ocurrió durante el wait de A.")
print(f"Sin create_task, hubiera sumado {1.0 + t_trabajo:.2f}s en lugar de {t_create_task:.2f}s")

## [EN CLASE] Sección 2: as_completed — procesar resultados conforme llegan

Con tareas de **duración variable**, `gather` espera al más lento antes de procesar cualquier resultado.
`as_completed` permite procesar cada resultado tan pronto como está disponible.

In [ ]:
import random

async def buscar_fuente(nombre: str, latencia: float) -> str:
    """Simula búsqueda en una fuente externa con latencia variable"""
    await asyncio.sleep(latencia)
    return f"resultado de {nombre} (tardó {latencia:.1f}s)"

fuentes = [
    ("Wikipedia", 0.5),
    ("ArXiv",     2.1),
    ("GitHub",    0.8),
    ("PubMed",    1.5),
    ("DuckDuck",  0.3),
]

# --- Con gather: espera el más lento ---
print("=== asyncio.gather: espera al más lento ===")
t0 = time.perf_counter()
resultados = await asyncio.gather(
    *[buscar_fuente(nombre, lat) for nombre, lat in fuentes]
)
t_gather = time.perf_counter() - t0
print(f"Todos los resultados disponibles a los {t_gather:.2f}s")
for r in resultados:
    print(f"  {r}")
print()

# --- Con as_completed: procesar conforme llegan ---
print("=== asyncio.as_completed: procesa conforme llegan ===")
t0 = time.perf_counter()
tareas = [asyncio.create_task(buscar_fuente(n, l)) for n, l in fuentes]

orden_llegada = []
async for tarea_completada in asyncio.as_completed(tareas):
    resultado = await tarea_completada
    t_llegada = time.perf_counter() - t0
    print(f"  t={t_llegada:.2f}s → {resultado}")
    orden_llegada.append(resultado)

print()
print("Observa: el orden de llegada es por latencia, no por orden de creación.")
print("El resultado más rápido (DuckDuck, 0.3s) llega primero aunque fue creado al final.")

---

## [TAREA] Sección 3: asyncio.Queue — productor-consumidor

Implementa el patrón productor-consumidor usando `asyncio.Queue`.

In [ ]:
# TAREA 3: Implementar el patrón productor-consumidor
#
# Escenario: un productor genera 10 peticiones de chatbot a ritmo de 2/s.
# 3 workers las procesan (cada una tarda ~0.5s).
# Mide: ¿cuándo termina todo si hay 1, 2 o 3 workers?

N_PETICIONES = 10
RITMO_PRODUCCION = 0.5   # segundos entre peticiones
T_PROCESAMIENTO = 0.5    # segundos que tarda cada petición

# TODO: implementa `productor(queue, n)` y `worker(queue, nombre)`
# Hint: usa `await queue.put(item)` y `await queue.get()`
# No olvides el sentinel para terminar los workers

# TODO: lanza con asyncio.gather y mide tiempos con 1, 2, 3 workers

print("TODO: implementa el patrón productor-consumidor")

## [TAREA] Sección 4: fire-and-forget — excepción silenciada vs tracked

In [ ]:
# TAREA 4: Demostrar fire-and-forget con excepción silenciada

async def tarea_que_falla():
    await asyncio.sleep(0.1)
    raise ValueError("¡Error en la tarea!")

# Anti-patrón: fire-and-forget sin tracking
print("=== Anti-patrón: fire-and-forget ===")
t = asyncio.create_task(tarea_que_falla())   # ← lanza la tarea
await asyncio.sleep(0.5)                      # ← la excepción ocurre, pero...
print("No se vio ningún error — ¡la excepción fue silenciada!")
# Nota: Python puede mostrar una advertencia al final, pero no interrumpe el programa
print()

# TODO: implementa el patrón correcto con callback done que logge errores
# Hint: t.add_done_callback(fn) donde fn recibe el future completado

print("TODO: implementa el tracking de excepciones con add_done_callback")

## [TAREA] Sección 5: Benchmark asyncio vs threading — I/O-bound

Compara asyncio y ThreadPoolExecutor para tareas I/O-bound.
Ambos deberían funcionar — ¿cuál es más rápido y más simple?

In [ ]:
import time
import requests  # síncrono
# import aiohttp  # si está disponible

# Simula I/O-bound síncrono (para ThreadPoolExecutor)
def io_bound_sync(duracion: float) -> str:
    time.sleep(duracion)   # simula requests.get, cursor.execute, etc.
    return "resultado"

N = 20
DUR = 0.2  # cada tarea tarda 0.2s

# --- ThreadPoolExecutor ---
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=N) as pool:
    futuros = [pool.submit(io_bound_sync, DUR) for _ in range(N)]
    resultados = [f.result() for f in futuros]
t_thread = time.perf_counter() - t0
print(f"ThreadPoolExecutor ({N} workers): {t_thread:.2f}s  (esperado ~{DUR:.1f}s)")

# TODO: implementa la versión con asyncio.gather
# Compara: ¿cuánto difieren? ¿En qué casos preferirías uno sobre el otro?
# Hint: asyncio.gather + asyncio.sleep (simula la versión async de io_bound_sync)

print("TODO: implementa y compara con asyncio.gather")

## [TAREA] Sección 6: threading vs multiprocessing — CPU-bound con el GIL en números

Confirma empíricamente la predicción del GIL: threading no escala para CPU-bound, multiprocessing sí.

In [ ]:
# IMPORTANTE: ProcessPoolExecutor requiere funciones picklables
# La función debe estar definida a nivel de módulo (no como lambda ni nested)

def tarea_cpu(n: int) -> int:
    """CPU-bound pura: wait(τᵢ) = ∅"""
    return sum(range(n))

N_TRABAJO = 20_000_000   # ajusta según tu máquina
N_WORKERS_LIST = [1, 2, 4, os.cpu_count()]

# --- Secuencial (baseline) ---
t0 = time.perf_counter()
for w in N_WORKERS_LIST:
    for _ in range(w):
        tarea_cpu(N_TRABAJO)
# Solo mide con N_WORKERS_LIST[-1] para baseline justo
n = N_WORKERS_LIST[-1]
t0 = time.perf_counter()
for _ in range(n): tarea_cpu(N_TRABAJO)
t_seq = time.perf_counter() - t0
print(f"Secuencial ({n} tareas): {t_seq:.2f}s (baseline)")
print()

# TODO: implementa el benchmark para threading y multiprocessing
# Para cada n_workers en [1, 2, 4, os.cpu_count()]:
#   - Mide tiempo con threading.Thread
#   - Mide tiempo con ProcessPoolExecutor
#   - Calcula speedup vs secuencial
# Grafica los resultados con matplotlib

print("TODO: implementa benchmark threading vs multiprocessing")

## [TAREA] Sección 7: joblib vs ProcessPoolExecutor — misma tarea

In [ ]:
try:
    from joblib import Parallel, delayed
    JOBLIB_OK = True
except ImportError:
    print("joblib no instalado — pip install joblib")
    JOBLIB_OK = False

if JOBLIB_OK:
    datos = list(range(1_000_000, 1_001_000))   # 1000 tareas
    
    # ProcessPoolExecutor
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=4) as pool:
        resultados_ppe = list(pool.map(tarea_cpu, datos))
    t_ppe = time.perf_counter() - t0
    print(f"ProcessPoolExecutor (4 workers): {t_ppe:.2f}s")
    
    # joblib
    t0 = time.perf_counter()
    resultados_jl = Parallel(n_jobs=4)(delayed(tarea_cpu)(n) for n in datos)
    t_jl = time.perf_counter() - t0
    print(f"joblib.Parallel (n_jobs=4):       {t_jl:.2f}s")
    
    assert resultados_ppe == resultados_jl, "resultados diferentes!"
    print(f"\nResultados idénticos: ✓")
    print(f"\n¿Cuándo preferirías joblib sobre ProcessPoolExecutor?")
    print("Hint: considera la interfaz, integración con numpy, y el backend 'loky'")

## [TAREA] Sección 8: Anti-patrón lambda (PicklingError)

In [ ]:
# TAREA 8: Reproducir el PicklingError con lambda en ProcessPoolExecutor

# Anti-patrón: lambda no puede ser serializada entre procesos
try:
    with ProcessPoolExecutor(max_workers=2) as pool:
        resultados = list(pool.map(lambda x: x**2, [1, 2, 3, 4]))
    print("¿Sin error? (puede pasar en algunos sistemas)")
    print(resultados)
except Exception as e:
    print(f"Error esperado: {type(e).__name__}: {e}")

print()

# TODO: aplica el fix con una función a nivel de módulo o functools.partial
# Verifica que produce el mismo resultado sin error

from functools import partial
# TODO: implementa al_cuadrado como función y usa partial para demostrar ambos fixes

## [TAREA] Sección 9: Pool por petición vs pool compartido — medir diferencia

In [ ]:
# TAREA 9: Comparar overhead de crear pool por petición vs pool compartido

N_PETICIONES = 20

# Anti-patrón: nuevo pool por petición
t0 = time.perf_counter()
for _ in range(N_PETICIONES):
    with ProcessPoolExecutor(max_workers=2) as pool:
        list(pool.map(tarea_cpu, [100_000, 100_000]))
t_pool_por_peticion = time.perf_counter() - t0
print(f"Pool por petición ({N_PETICIONES} veces): {t_pool_por_peticion:.2f}s")

# Correcto: pool compartido
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as pool_compartido:
    for _ in range(N_PETICIONES):
        list(pool_compartido.map(tarea_cpu, [100_000, 100_000]))
t_pool_compartido = time.perf_counter() - t0
print(f"Pool compartido ({N_PETICIONES} peticiones): {t_pool_compartido:.2f}s")

print(f"\nOverhead del anti-patrón: {t_pool_por_peticion/t_pool_compartido:.1f}× más lento")
print("El overhead es el costo de crear/destruir procesos N veces en lugar de una.")

## [TAREA] Sección 10 (opcional): run_in_executor — asyncio + ProcessPoolExecutor

Integra asyncio con ProcessPoolExecutor para manejar carga mixta (I/O + CPU).
Este es el patrón del chatbot v3 (M5b).

In [ ]:
# TAREA 10: Implementar M5b — asyncio + ProcessPoolExecutor
#
# Escenario: 10 peticiones llegan simultáneamente.
# Cada petición: consulta BD (I/O, 0.1s) + inferencia LLM local (CPU, 0.5s)
#
# Compara:
#   a) Todo en asyncio con time.sleep (bloquea el event loop durante inferencia)
#   b) asyncio para I/O + run_in_executor(ProcessPoolExecutor) para CPU
#
# ¿Cuánto mejora la versión b sobre la a?
# ¿Cómo se relaciona con la Ley de Amdahl?

N_USUARIOS = 10
T_IO = 0.1    # wait(τᵢ): consulta BD
T_CPU = 0.5   # exec bloqueante: inferencia LLM

def inferencia_local(historial):
    """CPU-bound: simula inferencia del LLM local"""
    time.sleep(T_CPU)   # usa time.sleep para simular blocking CPU work
    return f"respuesta para {historial}"

# TODO: implementa versión a (asyncio puro con time.sleep bloqueante)
# TODO: implementa versión b (asyncio + run_in_executor con ProcessPoolExecutor)
# TODO: mide tiempos y calcula la fracción secuencial S implícita

print("TODO: implementa M5b con asyncio + ProcessPoolExecutor")